In [ ]:
import pandas as pd
import numpy as np

# =========================
# 1. LEER ARCHIVO
# =========================
df = pd.read_excel("Profonanpe -  entidades.xlsx")

# Limpiar nombres de columnas
df.columns = df.columns.str.strip()

# =========================
# 2. FACT TABLE (PROYECTOS)
# =========================
fact_proyectos = df.drop(columns=['Departamento', 'Rol de Profonanpe'], errors='ignore') \
                   .drop_duplicates() \
                   .reset_index(drop=True)

# ID surrogate
fact_proyectos['IdProyecto'] = fact_proyectos.index + 1

# Renombrar clave
fact_proyectos = fact_proyectos.rename(columns={'Código': 'CodigoProyecto'})

# ===================================
# 2.1 TIPO DE CAMBIO Y DOLARIZACIÓN
# ===================================

tipo_cambio_usd = 3.394
tipo_cambio_eur = 3.962176

fact_proyectos['Tipo Cambio ($)'] = tipo_cambio_usd
fact_proyectos['Tipo Cambio (€)'] = tipo_cambio_eur

# Asegurar numérico
fact_proyectos['Monto del donante'] = pd.to_numeric(
    fact_proyectos['Monto del donante'], errors='coerce')

# =========================
# CONVERSIÓN A USD
# =========================

fact_proyectos['Monto Dolarizado'] = np.nan

# SOL → USD
mask_soles = fact_proyectos['Moneda'] == 'Soles'
fact_proyectos.loc[mask_soles, 'Monto Dolarizado'] = (
    fact_proyectos.loc[mask_soles, 'Monto del donante'] / tipo_cambio_usd
)

# EUR → SOL → USD
mask_euros = fact_proyectos['Moneda'] == 'Euros'
fact_proyectos.loc[mask_euros, 'Monto Dolarizado'] = (
    fact_proyectos.loc[mask_euros, 'Monto del donante'] * tipo_cambio_eur / tipo_cambio_usd
)

# USD → USD
mask_usd = fact_proyectos['Moneda'] == 'Dólares'
fact_proyectos.loc[mask_usd, 'Monto Dolarizado'] = (
    fact_proyectos.loc[mask_usd, 'Monto del donante']
)

# Redondeo
fact_proyectos['Monto Dolarizado'] = fact_proyectos['Monto Dolarizado'].astype(float).round(2)

# =========================
# 3. DEPARTAMENTOS
# =========================

puente_dep = df[['Código', 'Departamento']].dropna()

puente_dep['Departamento'] = puente_dep['Departamento'].str.split(';')
puente_dep = puente_dep.explode('Departamento')

puente_dep['Departamento'] = puente_dep['Departamento'].str.strip()

# Quitar tildes
puente_dep['Departamento'] = puente_dep['Departamento'].str.normalize('NFKD') \
    .str.encode('ascii', errors='ignore') \
    .str.decode('utf-8')

puente_dep = puente_dep.drop_duplicates().reset_index(drop=True)

# Dimensión
dim_departamentos = puente_dep[['Departamento']].drop_duplicates().reset_index(drop=True)
dim_departamentos['IdDepartamento'] = dim_departamentos.index + 1

# Relación
puente_dep = puente_dep.rename(columns={'Código': 'CodigoProyecto'})

puente_dep = puente_dep.merge(dim_departamentos, on='Departamento', how='left')

puente_dep = puente_dep.merge(
    fact_proyectos[['CodigoProyecto', 'IdProyecto']],
    on='CodigoProyecto',
    how='left'
)

bridge_departamento = puente_dep[['IdProyecto', 'IdDepartamento']]

# =========================
# 4. ROLES (NORMALIZADO)
# =========================

puente_rol = df[['Código', 'Rol de Profonanpe']].dropna().copy()

# Limpiar espacios
puente_rol['Rol de Profonanpe'] = puente_rol['Rol de Profonanpe'].str.strip()

# Separar
puente_rol['Rol de Profonanpe'] = puente_rol['Rol de Profonanpe'].str.split(';')

# Ordenar cada lista alfabéticamente
puente_rol['Rol de Profonanpe'] = puente_rol['Rol de Profonanpe'].apply(
    lambda x: sorted([i.strip().title() for i in x])
)

# Volver a unir con " y "
puente_rol['Rol de Profonanpe'] = puente_rol['Rol de Profonanpe'].str.join(' y ')

# Eliminar duplicados
puente_rol = puente_rol.drop_duplicates().reset_index(drop=True)

# Dimensión
dim_roles = puente_rol[['Rol de Profonanpe']].drop_duplicates().reset_index(drop=True)
dim_roles['IdRol'] = dim_roles.index + 1

# Relación
puente_rol = puente_rol.rename(columns={'Código': 'CodigoProyecto'})

puente_rol = puente_rol.merge(dim_roles, on='Rol de Profonanpe', how='left')

puente_rol = puente_rol.merge(
    fact_proyectos[['CodigoProyecto', 'IdProyecto']],
    on='CodigoProyecto',
    how='left'
)

bridge_roles = puente_rol[['IdProyecto', 'IdRol']]

# =========================
# 5. EXPORTAR
# =========================

with pd.ExcelWriter("modelo_estrella_proyectos.xlsx") as writer:
    fact_proyectos.to_excel(writer, sheet_name="Fact_Proyectos", index=False)
    dim_departamentos.to_excel(writer, sheet_name="Dim_Departamentos", index=False)
    bridge_departamento.to_excel(writer, sheet_name="Bridge_Proyecto_Departamento", index=False)
    dim_roles.to_excel(writer, sheet_name="Dim_Roles", index=False)
    bridge_roles.to_excel(writer, sheet_name="Bridge_Proyecto_Rol", index=False)

print("Modelo estrella generado correctamente")

Modelo estrella generado correctamente
